# Create California Energy Commission Awards

Creates awards for the **California Energy Commission (CEC)** from the agendas of its public
Business Meetings, where every CEC grant, loan and contract is approved. The agendas are docketed
as PDFs in the CEC e-filing system (dockets `YY-BUSMTG-01`,
https://efiling.energy.ca.gov/Lists/DocketLog.aspx?docketnumber=24-BUSMTG-01); each new agreement
is described as "*Recipient. Proposed resolution approving agreement EPC-23-035 with Recipient for a
$8,000,000 grant ... This project will ... (EPIC Funding)*". `scripts/local/cec_to_s3.py` takes the
latest revision of each meeting's agenda (65 meetings, May 2022 - Aug 2026; earlier agendas are not
in the docket system, and the CEC awards page refers pre-2022 awards to Public Records Act requests),
and parses every NEW agreement (amendments, terminations and novations skipped): 600 agreements.

**robots.txt:** the CEC's main site www.energy.ca.gov names no AI agents (only `User-agent: *` with
standard Drupal rules) and efiling.energy.ca.gov has no robots.txt. The CEC's separate EPIC project
database site, the Energy Innovation Showcase (www.energizeinnovation.fund), names ClaudeBot,
anthropic-ai, Claude-Web, Claude-SearchBot, Claude-User, GPTBot and other AI agents with
`Disallow: /`; it was **not used** (neither its pages nor its export). Coordinator decision
2026-10-01: the org-wide AI-block rule keys on the main site, so CEC stays in.

**Scope filter (2026-10-01 batch rule), applied here via `in_scope`:**
- **Kept, research (213):** grant agreements funded by EPIC (Electric Program Investment Charge,
  `EPC-`, 176), PIER Natural Gas / Gas R&D (`PIR-`, 29) and CRISP (Carbon Removal Innovation Support
  Program, `CRI-`, 8).
- **Kept and flagged (22):** demonstration / resource programmes that are not clearly research:
  Long-Duration Energy Storage demonstrations (`LDS-`, 12), INDIGO industrial decarbonization
  (`IND-`, 7), Geothermal Resources Development Account grants (`GEO-`, 3).
- **Dropped (365):** Clean Transportation Program / NEVI / ZVI / EVC vehicle and charger
  infrastructure, ECAA efficiency loans, FPIP, DEBA, DSGS, CERRI, building-decarbonisation
  incentives, offshore-wind port planning, IRA workforce grants, and the CEC's own support contracts,
  memberships and MOUs (numeric prefixes 100- to 800-, MOU-) even when paid from EPIC / Gas R&D money.

**235 awards**, approval dates 2022-05-11 to 2026-08-17, all grants.

**Fields:** `funder_award_id` = the agreement number (`EPC-23-035`, `PIR-22-003`), the form citing
papers carry (§2.1.1). `amount` = the grant amount in the agenda (USD, 97.9%; five CEQA-findings
items state no amount). `start_date` = the Business Meeting date (approval). `display_name` = the
project name where the agenda gives one (5 CEQA-findings items), else the first sentence of the
agenda's project description (leading "This project will" dropped, capped at 300 characters), else
the grant's purpose clause; `description` = purpose clause + project description. The agendas name
no people: `lead_investigator` carries only the recipient organisation as affiliation (country US),
names NULL. `funder_scheme` = the programme given by the agreement-number prefix (EPC- EPIC, PIR- PIER
Natural Gas R&D, CRI- CRISP, LDS- Long-Duration Energy Storage, IND- INDIGO, GEO- Geothermal Grant
and Loan Program); the agenda's own funding tag stays in the raw table (`funding_source`).
`landing_page_url` = the agenda PDF.

**Prerequisites:** run `scripts/local/cec_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/cec/cec_projects.parquet` (all 600 parsed agreements with `in_scope`
and `scope_reason`; §1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320308020`, California Energy
Commission, US; ror 05eaakg28 and doi 10.13039/100004805 read from `openalex.funders.funders`.

**Priority:** `573` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cec_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/cec/cec_projects.parquet`;

In [ ]:
%sql
SELECT in_scope, scope_reason, COUNT(*) as n
FROM openalex.awards.cec_raw
GROUP BY 1, 2 ORDER BY 1 DESC, 3 DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.cec_raw;

In [ ]:
%sql
SELECT agreement_number, recipient, amount, instrument, funding_source, meeting_date, title
FROM openalex.awards.cec_raw WHERE in_scope = 'True' LIMIT 10;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'California Energy Commission F4320308020 must have exactly one openalex.funders.funders row') as funder_ok,
       first(display_name) as display_name, first(ror_id) as ror_id, first(doi) as doi, first(country_code) as country_code
FROM openalex.funders.funders
WHERE funder_id = 4320308020;

## Step 2: Create CEC Awards Table (in-scope agreements only)

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cec_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320308020  -- California Energy Commission
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) as amount,
    CASE WHEN NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'grant' as funding_type,
    NULLIF(TRIM(g.programme), '') as funder_scheme,  -- from the agreement-number prefix
    'cec_business_meeting_agendas' as provenance,
    TRY_TO_DATE(g.meeting_date, 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    YEAR(TRY_TO_DATE(g.meeting_date, 'yyyy-MM-dd')) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN NULLIF(TRIM(g.recipient), '') IS NOT NULL THEN named_struct(
        'given_name', CAST(NULL AS STRING),
        'family_name', CAST(NULL AS STRING),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', TRIM(g.recipient),
            'country', 'US',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.cec_raw g
CROSS JOIN src_funder f
WHERE g.in_scope = 'True'
  AND g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- §2.1.1 shape check (fails the run BEFORE the insert): every key is a CEC agreement number of an
-- in-scope programme; ids unique; every in-scope row carried over.
SELECT
    assert_true(COUNT_IF(NOT funder_award_id RLIKE '^(EPC|PIR|CRI|LDS|IND|GEO)-[0-9]{2}-[0-9]{3}$') = 0,
                'funder_award_id is not an in-scope CEC agreement number') as shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') as ids_unique,
    assert_true(COUNT(*) = (SELECT COUNT(*) FROM openalex.awards.cec_raw WHERE in_scope = 'True'), 'in-scope rows dropped') as all_rows,
    COUNT(*) as total
FROM openalex.awards.cec_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cec_business_meeting_agendas' AND priority = 573;

-- Insert into openalex_awards_raw with priority.
-- Priority 573: direct-from-funder ingest of the CEC's Business Meeting agendas.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 573 outranks the priority-0
-- acknowledgement shells that share the EPC-/PIR- agreement numbers.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    573 as priority
FROM openalex.awards.cec_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.cec_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date,
       lead_investigator.affiliation.name as recipient
FROM openalex.awards.cec_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.cec_awards
GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.cec_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- §6.4a frequency check: no person names; recipient organisations top-20.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family,
       lead_investigator.affiliation.name AS org, COUNT(*) AS n
FROM openalex.awards.cec_awards
GROUP BY 1, 2, 3 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.4a title frequency check.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.cec_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- §6.3 / §6.7 coverage (expect 100% title/start_date/recipient, ~98% amount; USD only).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_recipient
FROM openalex.awards.cec_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs (same funder): these upgrade in CreateAwards dedup.
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(DISTINCT c.id) as distinct_awards
FROM openalex.awards.openalex_awards_raw c
JOIN openalex.awards.cec_awards t ON t.id = c.id
WHERE c.funder_id = 4320308020 AND c.priority = 0
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cec_business_meeting_agendas'
GROUP BY provenance, priority;